# 05 Figures (owner P3)
Builds every report figure and table from `results/*.csv`; nothing here changes results. Run after 02 (Ablation cells), 03 and 04.
Outputs: `figures/fig1..fig7*.png` and `results/tables/table*.csv`. Created 30 Oct by P1 + Claude, not yet run.

In [ ]:
# [Cell 1] Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# [Cell 2] Paths + folders + make the shared library importable
import os, sys
ROOT = '/content/drive/MyDrive/DSP_Deconv'
for d in ['lib', 'data/raw', 'data/processed', 'results', 'figures', 'configs']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
if f'{ROOT}/lib' not in sys.path:
    sys.path.insert(0, f'{ROOT}/lib')
print(os.listdir(ROOT))

In [ ]:
# [Cell 3] Packages + auto-reload edited library files
!pip -q install scikit-image tifffile pyyaml
%load_ext autoreload
%autoreload 2
import numpy as np, scipy, skimage, matplotlib.pyplot as plt
print('numpy', np.__version__, '| scipy', scipy.__version__, '| skimage', skimage.__version__)

In [ ]:
# [Cell 4] Global config (read only)
import yaml
cfg = yaml.safe_load(open(f'{ROOT}/configs/params.yaml'))
cfg

In [ ]:
# [Fig 0] imports, paths, shared plotting settings
import os, yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from forward import *; from data import *; from baselines import *; from proposed import *; from metrics import *
tuned = yaml.safe_load(open(f'{ROOT}/configs/tuned.yaml'))
RES, FIG, TAB = f'{ROOT}/results', f'{ROOT}/figures', f'{ROOT}/results/tables'
os.makedirs(TAB, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10})
ORDER = ['wiener_const', 'tikhonov_gcv', 'rl', 'unsup_wiener', 'proposed', 'oracle_wiener']
COLOR = {m: f'C{i}' for i, m in enumerate(ORDER)}

gt, _ = fmd_pair('WideField_BPAE_B', 1)          # dev image: used for illustrations only
x_dev = center_crop(norm_pair(gt)[0], cfg['crop'])

def mean_std(d, keys, col, nd=2, order=ORDER):
    """'mean ± std' strings, one column per method."""
    g = d.groupby(keys + ['method'])[col]
    t = (g.mean().round(nd).astype(str) + ' ± ' + g.std().round(nd).astype(str)).unstack()
    return t[[m for m in order if m in t.columns]]

In [ ]:
# [Fig 1] forward model: clean x, PSF, observed y, and |H| along one frequency axis for each PSF width
psf = gaussian_psf(psf_size(2.5), 2.5)
y = add_gaussian(blur(x_dev, psf), 0.02, np.random.default_rng(0))
fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
ax[0].imshow(x_dev, cmap='gray'); ax[0].set_title('clean x (WideField dev crop)')
ax[1].imshow(psf, cmap='magma'); ax[1].set_title('Gaussian PSF, σ = 2.5 px')
ax[2].imshow(np.clip(y, 0, 1), cmap='gray'); ax[2].set_title('observed y, σ_n = 0.02')
for a in ax[:3]: a.axis('off')
f = np.fft.rfftfreq(cfg['crop'])
for s in cfg['psf_sigmas']:
    H = np.abs(psf2otf(gaussian_psf(psf_size(s), s), (cfg['crop'], cfg['crop'])))[0, :len(f)]
    ax[3].semilogy(f, H, label=f'σ = {s} px')
ax[3].axhline(np.sqrt(cfg['eps_band']), ls=':', c='k', label='noise-only band starts')
ax[3].set_xlabel('frequency (cycles/px)'); ax[3].set_ylabel('|H|'); ax[3].set_ylim(1e-6, 2); ax[3].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/fig1_forward_model.png'); plt.show()

In [ ]:
# [Fig 2] the key figure: estimated lambda(w) vs oracle noise-to-signal ratio, at low and high noise
w = freq_grid(x_dev.shape); bins = np.linspace(0, 0.5, 51); fc = (bins[:-1] + bins[1:]) / 2
radial = lambda M: np.array([np.median(M[(w >= a) & (w < b)]) for a, b in zip(bins[:-1], bins[1:])])
psf = gaussian_psf(psf_size(2.5), 2.5)
H2 = np.abs(psf2otf(psf, x_dev.shape)) ** 2
Sx = np.abs(np.fft.fft2(x_dev)) ** 2 / x_dev.size
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, s_n in zip(ax, [0.005, 0.05]):
    y = add_gaussian(blur(x_dev, psf), s_n, np.random.default_rng(0))
    _, info = adaptive_wiener(y, psf, return_info=True)
    a.loglog(fc, radial(info['lam']), label='estimated λ(ω), proposed')
    a.loglog(fc, radial(s_n ** 2 / (Sx + 1e-12)), label='oracle NSR σ²/S_x(ω)')
    a.loglog(fc, radial(H2), '--', c='gray', label='|H(ω)|²')
    a.axhline(tuned['nsr'], c='C3', ls=':', label='tuned constant NSR')
    a.set_title(f'σ_n = {s_n}: σ̂ = {np.sqrt(info["sigma2"]):.4f}, β = {info["beta"]:.2f}')
    a.set_xlabel('frequency (cycles/px)')
ax[0].set_ylabel('regularizer value'); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/fig2_lambda.png'); plt.show()

In [ ]:
# [Fig 3 + Tables 1-2] Gaussian noise: PSNR vs noise level per PSF width (mean ± std over 8 crops x 5 seeds)
df = pd.read_csv(f'{RES}/synthetic_gauss_test.csv'); df = df[df.method != 'inverse']
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for a, s in zip(ax, cfg['psf_sigmas']):
    d = df[df.psf == s].groupby(['method', 'noise']).psnr.agg(['mean', 'std']).reset_index()
    for m in ORDER:
        r = d[d.method == m]
        a.errorbar(r.noise, r['mean'], yerr=r['std'], marker='o', capsize=3, color=COLOR[m], label=m,
                   ls='--' if m == 'oracle_wiener' else '-')
    a.set_xscale('log'); a.set_title(f'PSF σ = {s} px'); a.set_xlabel('noise σ_n'); a.grid(alpha=0.3)
ax[0].set_ylabel('PSNR (dB)'); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/fig3_psnr_vs_noise.png'); plt.show()

mean_std(df, ['psf', 'noise'], 'psnr').to_csv(f'{TAB}/table1_gauss_psnr.csv')
mean_std(df, ['psf', 'noise'], 'ssim', 3).to_csv(f'{TAB}/table1b_gauss_ssim.csv')
df.groupby('method')[['psnr', 'ssim', 'lsd', 'ms']].agg(['mean', 'std']).round(3).to_csv(f'{TAB}/table2_overall.csv')
mean_std(df, ['psf', 'noise'], 'psnr')

In [ ]:
# [Fig 3b + Table 3] Poisson-Gaussian noise (from 03 S6 Cell B2), x-axis = photons at full brightness
p = Path(f'{RES}/synthetic_pg_test.csv')
if p.exists():
    pg = pd.read_csv(p)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for a, s in zip(ax, cfg['psf_sigmas']):
        d = pg[pg.psf == s].groupby(['method', 'peak']).psnr.agg(['mean', 'std']).reset_index()
        for m in ORDER:
            r = d[d.method == m]
            a.errorbar(r.peak, r['mean'], yerr=r['std'], marker='o', capsize=3, color=COLOR[m], label=m,
                       ls='--' if m == 'oracle_wiener' else '-')
        a.set_xscale('log'); a.set_title(f'PSF σ = {s} px'); a.set_xlabel('peak photons (more = less noise)'); a.grid(alpha=0.3)
    ax[0].set_ylabel('PSNR (dB)'); ax[0].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f'{FIG}/fig3b_psnr_poisson.png'); plt.show()
    mean_std(pg, ['psf', 'peak'], 'psnr').to_csv(f'{TAB}/table3_pg_psnr.csv')
else:
    print('synthetic_pg_test.csv not found: run 03_experiments S6 Cell B2 first')

In [ ]:
# [Fig 4 + Table 4] edge phantom: row-averaged edge profiles for one noise realization + metrics from edge_test.csv
x = edge_phantom(); psf = gaussian_psf(psf_size(2.5), 2.5); s_n = 0.02
y = add_gaussian(blur(x, psf), s_n, np.random.default_rng(0))
outs = {'observed': y,
        'wiener_const': wiener_const(y, psf, tuned['nsr']),
        'tikhonov_gcv': tikhonov_gcv(y, psf),
        'rl': rl(y, psf, tuned['rl_iter']),
        'unsup_wiener': unsup_wiener(y, psf),
        'proposed': adaptive_wiener(y, psf),
        'oracle_wiener': wiener_oracle(y, psf, x, s_n)}
n = x.shape[1]; cols = np.arange(n // 2 - 20, n // 2 + 20)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cols, x[:, cols].mean(0), 'k:', label='true edge')
for m, im in outs.items():
    ax.plot(cols, im[:, cols].mean(0), color=COLOR.get(m, 'gray'), label=m)
ax.set_xlabel('column (px)'); ax.set_ylabel('row-averaged intensity'); ax.set_title(f'PSF σ = 2.5 px, σ_n = {s_n}')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIG}/fig4_edge_profiles.png'); plt.show()

e = pd.read_csv(f'{RES}/edge_test.csv')
t4 = e.groupby('method')[['fwhm_px', 'mtf50', 'overshoot_pct', 'noise_amp']].agg(['mean', 'std']).round(3)
t4.to_csv(f'{TAB}/table4_edge.csv')
e.groupby(['noise', 'method'])[['fwhm_px', 'mtf50', 'overshoot_pct', 'noise_amp']].mean().round(3).to_csv(f'{TAB}/table4b_edge_by_noise.csv')
t4

In [ ]:
# [Fig 5 + Table 5] robustness: PSF-mismatch curve and border error maps (needs 03 S7 CSVs)
mis = pd.read_csv(f'{RES}/psf_mismatch.csv'); bnd = pd.read_csv(f'{RES}/boundary.csv')
psf = gaussian_psf(psf_size(2.5), 2.5)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
d = mis.groupby(['method', 'error_pct']).psnr.agg(['mean', 'std']).reset_index()
for m in d.method.unique():
    r = d[d.method == m]
    ax[0].errorbar(r.error_pct, r['mean'], yerr=r['std'], marker='o', capsize=3, color=COLOR[m], label=m)
ax[0].axvline(0, c='k', ls=':'); ax[0].set_xlabel('PSF width error (%)'); ax[0].set_ylabel('PSNR (dB)')
ax[0].set_title('true PSF σ = 2.5 px, σ_n = 0.01'); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

x = quadrants(norm_pair(fmd_pair('Confocal_BPAE_B', 19)[0])[0])[0]
y = add_gaussian(blur(x, psf, boundary='reflect'), 0.01, np.random.default_rng(0))
for a, (title, xh) in zip(ax[1:], [('no padding', adaptive_wiener(y, psf)), ('mirror padding', padded(adaptive_wiener, y, psf))]):
    xc = np.clip(xh, 0, 1)
    im = a.imshow(np.abs(xc - x), cmap='inferno', vmin=0, vmax=0.3)
    a.set_title(f'|error|, proposed, {title}: {psnr(x, xc):.1f} dB'); a.axis('off')
fig.colorbar(im, ax=ax[2], fraction=0.046)
plt.tight_layout(); plt.savefig(f'{FIG}/fig5_robustness.png'); plt.show()

d.pivot_table(index='error_pct', columns='method', values='mean').round(2).to_csv(f'{TAB}/table5_psf_mismatch.csv')
t5 = bnd.groupby('method')[['psnr_plain', 'psnr_padded']].agg(['mean', 'std']).round(2)
t5.to_csv(f'{TAB}/table5b_boundary.csv'); t5

In [ ]:
# [Fig 6 + Tables 6-7] real data summaries (FMD figures themselves come from 04_real_data)
fmd = pd.read_csv(f'{RES}/real_fmd.csv'); oct_df = pd.read_csv(f'{RES}/real_oct.csv')
fmd.pivot_table(index=['subset', 'psf'], columns='method', values=['noise_gain', 'mtf50']).round(3).to_csv(f'{TAB}/table6_real_fmd.csv')
oct_df.groupby('method')[['background_noise_ratio', 'lateral_mtf50']].agg(['mean', 'std', 'max']).round(3).to_csv(f'{TAB}/table7_real_oct.csv')

methods = ['wiener_const', 'tikhonov_gcv', 'proposed']
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([oct_df[oct_df.method == m].background_noise_ratio for m in methods])
ax.set_xticks(range(1, len(methods) + 1), methods)
ax.axhline(1, c='k', ls=':'); ax.set_yscale('log')
ax.set_ylabel('background noise after / before'); ax.set_title(f'OCT, {oct_df.image.nunique()} B-scans (qualitative)')
plt.tight_layout(); plt.savefig(f'{FIG}/fig6_oct_noise.png'); plt.show()
oct_df.groupby('method')[['background_noise_ratio', 'lateral_mtf50']].agg(['mean', 'std', 'max']).round(3)

In [ ]:
# [Fig 7 + Tables 8-9] ablation (needs 02 Ablation cells): PSNR change of each variant relative to the full method
abl = pd.read_csv(f'{RES}/ablation.csv')
p = abl.pivot_table(index=['img', 'psf', 'noise', 'seed'], columns='variant', values='psnr')
gain = p[['oracle_sigma', 'oracle_Sx', 'const_lambda']].sub(p['full'], axis=0)
t8 = gain.groupby(level='psf').agg(['mean', 'std']).round(2)
t8.to_csv(f'{TAB}/table8_ablation.csv')
gain.groupby(level='psf').mean().plot.bar(figsize=(7, 4), rot=0)
plt.axhline(0, c='k', lw=0.8); plt.ylabel('PSNR minus full method (dB)'); plt.xlabel('PSF σ (px)')
plt.title('Ablation: >0 means that component limits the method')
plt.tight_layout(); plt.savefig(f'{FIG}/fig7_ablation.png'); plt.show()

full = abl[abl.variant == 'full']
full.groupby(['psf', 'noise'])[['sigma_ratio', 'beta']].agg(['mean', 'std']).round(3).to_csv(f'{TAB}/table9_estimator_accuracy.csv')
t8

In [ ]:
# [Tables 10] statistics table (from 03 S6 Cell D) and a list of every file this notebook wrote
print(pd.read_csv(f'{RES}/wilcoxon.csv').round(4).to_string(index=False))
print(sorted(os.listdir(FIG))); print(sorted(os.listdir(TAB)))